In [2]:
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio

fnames = sorted(f for f in os.listdir('reports') if f.endswith('.tsv'))
big_df = pd.concat(
    [pd.read_csv(os.path.join('reports', f), sep='\t') for f in fnames],
    ignore_index=True,
)
big_df['step'] = big_df['checkpoint'].str.extract(r'(\d+)$').astype(int)
big_df['step_plus1'] = big_df['step'] + 1          # lets step 0 appear on a log axis
big_df = big_df.sort_values(['subject', 'step', 'layer', 'roi'])

for subject in big_df['subject'].unique():
    d = big_df[(big_df['subject'] == subject)].copy()
    d['step'] = d['checkpoint'].str.extract(r'(\d+)$').astype(int)
    d['step_plus1'] = d['step'] + 1   # lets step 0 appear on a log axis

    fig = px.line(
        d.sort_values('step'), x='step_plus1', y='median_r',
        color='layer', facet_col='roi', markers=True, log_x=True,
    )
    pio.write_html(fig, f'html/median_{subject}.html')
    
    base = (d[d['step'] == 0][['layer', 'roi', 'median_r']]
        .rename(columns={'median_r': 'base_r'}))
    d = d.merge(base, on=['layer', 'roi'])
    d['delta_r'] = d['median_r'] - d['base_r']

    fig = px.line(d.sort_values('step'), x='step_plus1', y='delta_r',
            color='layer', facet_col='roi', markers=True, log_x=True)
    pio.write_html(fig, f'html/delta_{subject}.html')
    
    d = big_df[big_df['checkpoint'].isin(['step_0', 'step_2000'])]
    print(d[d['layer'] == 12].pivot_table(index=['subject', 'roi'], columns='checkpoint', values='median_r').round(3))

checkpoint        step_0  step_2000
subject roi                        
CSI1    EarlyVis   0.106      0.136
        LOC        0.050      0.093
        OPA        0.070      0.120
        PPA        0.058      0.116
        RSC        0.079      0.135
CSI2    EarlyVis   0.076      0.095
        LOC        0.048      0.080
        OPA        0.056      0.087
        PPA        0.076      0.129
        RSC        0.045      0.075
CSI3    EarlyVis   0.051      0.073
        LOC        0.029      0.060
        OPA        0.041      0.084
        PPA        0.050      0.108
        RSC        0.055      0.104
CSI4    EarlyVis   0.077      0.096
        LOC        0.028      0.060
        OPA        0.036      0.098
        PPA        0.044      0.109
        RSC        0.042      0.119
checkpoint        step_0  step_2000
subject roi                        
CSI1    EarlyVis   0.106      0.136
        LOC        0.050      0.093
        OPA        0.070      0.120
        PPA        0.058    